#Paquetes necesarios

In [1]:
import cv2  
import math 

from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\paull\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Modelos preentrenados, visualizando con las utilidades de ultralytics

In [2]:
# Carga del modelo
#model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolo11n-seg.pt') #Máscaras
model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model(filename, show=True)

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 271.7ms
video 1/1 (frame 2/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 15.8ms
video 1/1 (frame 3/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 (no detections), 25.2ms
vi

In [10]:
from collections import defaultdict
import numpy as np

# Carga del modelo, descarga en disco si no está presente en la carpeta
model = YOLO('yolo11n.pt') #Contenedores

# Etiqueta de las distintas clases
classNames = ["person", "bicycle", "car"]


# Captura desde la webcam
vid = cv2.VideoCapture("C0142.MP4")
track_history = defaultdict(lambda: [])

# Para guardar el vídeo procesado para ello necesitamos el códec, los fps y el tamaño del vídeo original
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = vid.get(cv2.CAP_PROP_FPS)
width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Guardar el vídeo procesado
out = cv2.VideoWriter("C0142_procesado.mp4", fourcc, fps, (width, height))

# Inicializamos los contadores y los sets para almacenar los IDs contados
contadores = {"person": 0, "bicycle": 0, "car": 0}
ids_contados = {"person": set(), "bicycle": set(), "car": set()}

while(True):      
    # fotograma a fotograma
    ret, img = vid.read()
  
    # si hay imagen válida
    if ret:  
        # Seguimiento, con persistencia entre fotogramas
        results = model.track(img, persist=True, classes = [0,1,2])
        
        # Para cada detección
        for r in results:
            boxes = r.boxes

            for box in boxes:
                # Contenedor
                x1, y1, x2, y2 = box.xyxy[0]
                x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)

                # Aplicar blur
                roi = img[y1:y2, x1:x2]
                roi_blur = cv2.GaussianBlur(roi, (51, 51), 0)
                img[y1:y2, x1:x2] = roi_blur

                # Etiqueta de seguimiento
                if box.id is not None:
                    track_id = str(int(box.id[0].tolist()))
                else:
                    track_id = ''
                
                # Confianza
                confidence = math.ceil((box.conf[0]*100))/100
                print("Confianza --->",confidence)

                # Clase
                cls = int(box.cls[0])
                print("Clase -->", classNames[cls])

       # Actualiza los contadores si el ID de seguimiento no ha sido contado previamente
                if box.id is not None:
                    nombre_clase = classNames[cls]
                    if track_id not in ids_contados[nombre_clase]:
                        ids_contados[nombre_clase].add(track_id)
                        contadores[nombre_clase] += 1

                # Convierte identificador numérico de clase a un color RGB
                escala = int((cls / len(classNames)) * 255 * 3)
                if escala >= 255*2:
                    R = 255
                    G = 255
                    B = escala - 255*2
                else:
                    if escala >= 255:
                        R = 255
                        G = escala - 255
                        B = 0
                    else:
                        R = escala
                        G = 0
                        B = 0

                # Dibuja el contenedor y clase
                cv2.rectangle(img, (x1, y1), (x2, y2), (R, G, B), 3)
                cv2.putText(img, track_id + ' ' + classNames[cls] , [x1, y1], cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, B), 2)

       # Dibuja los contadores en la esquina superior izquierda
        cv2.putText(img, f'Personas: {contadores["person"]}', (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        cv2.putText(img, f'Bicicletas: {contadores["bicycle"]}', (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        cv2.putText(img, f'Coches: {contadores["car"]}', (10, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

       # Guardar el fotograma procesado en el vídeo de salida
        out.write(img)

        # Muestra fotograma
        cv2.imshow('Vid', img)
    
    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break
  
# Libera el objeto de captura
vid.release()
# Libera el objeto de escritura de vídeo
out.release()
# Destruye ventanas
cv2.destroyAllWindows()



0: 384x640 1 car, 111.1ms
Speed: 8.6ms preprocess, 111.1ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.85
Clase --> car

0: 384x640 1 car, 116.6ms
Speed: 3.8ms preprocess, 116.6ms inference, 4.0ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 117.9ms
Speed: 4.9ms preprocess, 117.9ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 140.4ms
Speed: 4.9ms preprocess, 140.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.84
Clase --> car

0: 384x640 1 car, 112.9ms
Speed: 4.9ms preprocess, 112.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.82
Clase --> car

0: 384x640 1 car, 116.5ms
Speed: 4.2ms preprocess, 116.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Confianza ---> 0.81
Clase --> car

0: 384x640 1 car, 103.6ms
Speed: 7.6ms prepro

In [26]:
from ultralytics import YOLO
import cv2

model = YOLO("yolo11n.pt")

classNames = ["person", "bicycle", "car"]

cap = cv2.VideoCapture("C0142.MP4")

# obtener las caracteristicas del video
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# crear el video de salida
out = cv2.VideoWriter(
    "C0142_direccion.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

# linea roja para coches y bicicletas
linea_vehiculos_y = int(height * 430 / 928)
linea_vehiculos_x1 = int(width * 500 / 1650)
linea_vehiculos_x2 = width - 5

# linea azul para personas
# esta linea atraviesa la acera en lugar de seguir su direccion
# linea azul para personas
persona_x1 = int(width * 450 / 1650)
persona_y1 = int(height * 325 / 928)

persona_x2 = int(width * 500 / 1650)
persona_y2 = int(height * 390 / 928)

# margen de seguridad
margen = 12

# contadores de direccion
flujo = {
    "person": {"acercandose": 0, "alejandose": 0},
    "bicycle": {"acercandose": 0, "alejandose": 0},
    "car": {"acercandose": 0, "alejandose": 0}
}

# guardar el ultimo lado de cada objeto
posiciones = {}

# guardar la ultima posicion de las personas
posiciones_personas = {}

# evitar contar dos veces el mismo objeto
contados = set()

while cap.isOpened():
    ret, img = cap.read()

    if not ret:
        break

    # detectar y seguir los objetos
    results = model.track(
        img,
        persist=True,
        classes=[0, 1, 2],
        verbose=False
    )

    for box in results[0].boxes:

        # comprobar que el objeto tiene identificador
        if box.id is None:
            continue

        # obtener coordenadas, identificador y clase
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        track_id = int(box.id[0])
        cls = int(box.cls[0])
        nombre = classNames[cls]

        # utilizar el centro inferior de la caja
        cx = (x1 + x2) // 2
        cy = y2

        # analizar coches y bicicletas
        if nombre in ["car", "bicycle"]:

            # comprobar si esta por encima o debajo de la linea roja
            if linea_vehiculos_x1 <= cx <= linea_vehiculos_x2:

                if cy < linea_vehiculos_y - margen:
                    lado_actual = -1
                elif cy > linea_vehiculos_y + margen:
                    lado_actual = 1
                else:
                    lado_actual = 0

                # guardar la primera posicion conocida
                if track_id not in posiciones:
                    if lado_actual != 0:
                        posiciones[track_id] = lado_actual

                # comprobar si ha cruzado la linea
                elif track_id not in contados:

                    lado_anterior = posiciones[track_id]

                    if lado_actual != 0:

                        # de arriba hacia abajo: se acerca
                        if lado_anterior == -1 and lado_actual == 1:
                            flujo[nombre]["acercandose"] += 1
                            contados.add(track_id)

                        # de abajo hacia arriba: se aleja
                        elif lado_anterior == 1 and lado_actual == -1:
                            flujo[nombre]["alejandose"] += 1
                            contados.add(track_id)

                        # actualizar el ultimo lado conocido
                        posiciones[track_id] = lado_actual

        # analizar personas en la acera
        elif nombre == "person":

            # guardar la posicion inicial del peaton
            if track_id not in posiciones_personas:
                posiciones_personas[track_id] = (cx, cy)

            elif track_id not in contados:

                # recuperar la posicion anterior
                anterior_x, anterior_y = posiciones_personas[track_id]

                # calcular en que lado de la linea esta el peaton
                lado_anterior = (
                    (persona_x2 - persona_x1) * (anterior_y - persona_y1)
                    - (persona_y2 - persona_y1) * (anterior_x - persona_x1)
                )

                lado_actual = (
                    (persona_x2 - persona_x1) * (cy - persona_y1)
                    - (persona_y2 - persona_y1) * (cx - persona_x1)
                )

                # comprobar si ha cambiado de lado
                if lado_anterior * lado_actual < 0:

                    # comprobar que el cruce se produce cerca del segmento
                    medio_x = (anterior_x + cx) // 2
                    medio_y = (anterior_y + cy) // 2

                    cerca_linea = (
                        min(persona_x1, persona_x2) - 30 <= medio_x <= max(persona_x1, persona_x2) + 30
                        and min(persona_y1, persona_y2) - 30 <= medio_y <= max(persona_y1, persona_y2) + 30
                    )

                    if cerca_linea:

                        # de un lado hacia el otro: se acerca
                        if lado_anterior > 0 and lado_actual < 0:
                            flujo["person"]["alejandose"] += 1

                        else:
                            flujo["person"]["acercandose"] += 1

                        contados.add(track_id)

                # actualizar la posicion anterior
                posiciones_personas[track_id] = (cx, cy)

        # dibujar la deteccion y el identificador
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            img,
            f"{nombre} ID:{track_id}",
            (x1, y1 - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    # dibujar la linea roja de vehiculos
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y),
        (linea_vehiculos_x2, linea_vehiculos_y),
        (0, 0, 255),
        3
    )

    # dibujar los limites del margen de seguridad
    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y - margen),
        (linea_vehiculos_x2, linea_vehiculos_y - margen),
        (0, 255, 255),
        1
    )

    cv2.line(
        img,
        (linea_vehiculos_x1, linea_vehiculos_y + margen),
        (linea_vehiculos_x2, linea_vehiculos_y + margen),
        (0, 255, 255),
        1
    )

    # dibujar la linea azul de personas
    cv2.line(
        img,
        (persona_x1, persona_y1),
        (persona_x2, persona_y2),
        (255, 0, 0),
        3
    )

    # mostrar los contadores
    y = 30

    for nombre in classNames:
        texto = (
            f"{nombre}: "
            f"acercandose {flujo[nombre]['acercandose']} "
            f"alejandose {flujo[nombre]['alejandose']}"
        )

        cv2.putText(
            img,
            texto,
            (10, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 0, 255),
            2
        )

        y += 25

    # guardar y mostrar el fotograma
    out.write(img)
    cv2.imshow("Analisis de direccion", img)

    # Detenemos pulsado ESC
    if cv2.waitKey(20) == 27:
        break

# liberar recursos
cap.release()
out.release()
cv2.destroyAllWindows()

# mostrar los resultados finales
print("Resultados del flujo:")

for nombre in classNames:
    print(nombre, flujo[nombre])

Resultados del flujo:
person {'acercandose': 1, 'alejandose': 7}
bicycle {'acercandose': 0, 'alejandose': 0}
car {'acercandose': 16, 'alejandose': 11}


Intregración con seguimiento (tracking)
!!!!!!!!!Nota: he tenido que bajar a la versión de python 3.9.5 e instalar lap con pip install lap

In [14]:
# Carga del modelo
model = YOLO('yolo11n.pt') #Contenedores
#model = YOLO('yolov11n-seg.pt') #Máscaras
#model = YOLO('yolo11n-pose.pt')  #Pose

#Para un vídeo 
filename = "TGC23_PdH_C0056cut.mp4"
results = model.track(source=filename, show=True)  # BoT-SORT tracker (por defecto)
#results = model.track(source=filename, show=True, tracker="bytetrack.yaml")  # ByteTrack tracker

cv2.destroyAllWindows()


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 10.6ms
video 1/1 (frame 2/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 11.9ms
video 1/1 (frame 3/375) f:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\TGC23_PdH_C0056cut.mp4: 384x640 4 persons, 1 bird, 13.8ms
video

In [1]:

import json
import random
import shutil
from pathlib import Path

# localizar el dataset original
posibles_rutas = [Path("../matriculas"), Path("matriculas")]
raiz_dataset = None

for ruta in posibles_rutas:
    if (ruta / "images").is_dir() and (ruta / "annotations").is_dir():
        raiz_dataset = ruta
        break

if raiz_dataset is None:
    raise FileNotFoundError("No se encuentra el dataset original")

imagenes_dir = raiz_dataset / "images"
anotaciones_dir = raiz_dataset / "annotations"
salida_dir = raiz_dataset / "dataset_yolo"

print("Dataset original:", raiz_dataset.resolve())

# buscar imagenes y json
extensiones = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

imagenes = {}
for ext in extensiones:
    for archivo in imagenes_dir.glob("*" + ext):
        imagenes[archivo.stem] = archivo

anotaciones = {
    archivo.stem: archivo
    for archivo in anotaciones_dir.glob("*.json")
}

# comprobar que cada imagen tenga su json
comunes = sorted(set(imagenes) & set(anotaciones))

print("\nImágenes encontradas:", len(imagenes))
print("JSON encontrados:", len(anotaciones))
print("Parejas encontradas:", len(comunes))
print("Imágenes sin JSON:", len(set(imagenes) - set(anotaciones)))
print("JSON sin imagen:", len(set(anotaciones) - set(imagenes)))

# preparar las anotaciones validas
datos = []
json_invalidos = 0
anotaciones_descartadas = 0

for nombre in comunes:
    try:
        with open(anotaciones[nombre], "r", encoding="utf-8") as f:
            contenido = json.load(f)

        if not isinstance(contenido, list):
            json_invalidos += 1
            continue

        etiquetas = []

        for objeto in contenido:
            if not isinstance(objeto, dict):
                anotaciones_descartadas += 1
                continue

            rect = objeto.get("rectangle")

            if not isinstance(rect, list) or len(rect) != 4:
                anotaciones_descartadas += 1
                continue

            try:
                x, y, w, h = map(float, rect)
            except (TypeError, ValueError):
                anotaciones_descartadas += 1
                continue

            if not all(0 <= v <= 1 for v in [x, y, w, h]) or w <= 0 or h <= 0:
                anotaciones_descartadas += 1
                continue

            etiquetas.append(f"0 {x:.10f} {y:.10f} {w:.10f} {h:.10f}")

        if etiquetas:
            datos.append((imagenes[nombre], etiquetas))

    except (json.JSONDecodeError, OSError, TypeError):
        json_invalidos += 1

# mezclar y dividir el dataset
random.seed(42)
random.shuffle(datos)

total = len(datos)
n_train = int(total * 0.70)
n_val = int(total * 0.20)

conjuntos = {
    "train": datos[:n_train],
    "val": datos[n_train:n_train + n_val],
    "test": datos[n_train + n_val:]
}

# crear las carpetas de salida
if salida_dir.exists():
    shutil.rmtree(salida_dir)

for conjunto, elementos in conjuntos.items():
    (salida_dir / "images" / conjunto).mkdir(parents=True, exist_ok=True)
    (salida_dir / "labels" / conjunto).mkdir(parents=True, exist_ok=True)

    for imagen, etiquetas in elementos:
        shutil.copy2(imagen, salida_dir / "images" / conjunto / imagen.name)

        etiqueta_file = salida_dir / "labels" / conjunto / (imagen.stem + ".txt")

        with open(etiqueta_file, "w", encoding="utf-8") as f:
            f.write("\n".join(etiquetas))

# crear configuracion de YOLO
yaml = f"""path: {salida_dir.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['matricula']
"""

with open(salida_dir / "data.yaml", "w", encoding="utf-8") as f:
    f.write(yaml)

# mostrar resultados
print("\nDataset preparado correctamente")
print("Imágenes válidas:", total)
print("JSON inválidos:", json_invalidos)
print("Anotaciones descartadas:", anotaciones_descartadas)

for conjunto, elementos in conjuntos.items():
    print(f"{conjunto}: {len(elementos)} imágenes")

print("\nDataset guardado en:", salida_dir.resolve())


Dataset original: C:\Users\David\OneDrive\Documents\VC_P4\VC_P4\matriculas

Imágenes encontradas: 1456
JSON encontrados: 1455
Parejas encontradas: 1455
Imágenes sin JSON: 1
JSON sin imagen: 0


KeyboardInterrupt: 

In [22]:

import torch
from ultralytics import YOLO
from pathlib import Path

# rutas
dataset_dir = Path("../matriculas/dataset_yolo")
modelo_path = Path("yolo11n.pt")

# comprobar que existen los archivos
if not dataset_dir.exists():
    raise FileNotFoundError("No se encuentra el dataset")

if not modelo_path.exists():
    raise FileNotFoundError("No se encuentra yolo11n.pt")

# seleccionar dispositivo
device = 0 if torch.cuda.is_available() else "cpu"

print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

# cargar modelo preentrenado
model = YOLO(str(modelo_path))

# entrenar detector de matriculas
resultados = model.train(
    data=str(dataset_dir / "data.yaml"),
    epochs=100,
    patience=20,
    imgsz=640,
    batch=8,
    device=device,
    workers=0,
    project="../matriculas/runs",
    name="train_matriculas",
    exist_ok=True,
    plots=True
)


Dispositivo: NVIDIA GeForce RTX 2060
New https://pypi.org/project/ultralytics/8.4.198 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.191  Python-3.9.21 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=..\matriculas\dataset_yolo\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=

In [23]:

from pathlib import Path
from ultralytics import YOLO

# localizar el mejor modelo entrenado
candidatos = list(Path("..").rglob("train_matriculas/weights/best.pt"))

if not candidatos:
    candidatos = list(Path(".").rglob("train_matriculas/weights/best.pt"))

if not candidatos:
    raise FileNotFoundError("No se encuentra best.pt")

modelo_path = candidatos[0]

# localizar el dataset
dataset_dir = Path("../matriculas/dataset_yolo")

if not dataset_dir.exists():
    dataset_dir = Path("matriculas/dataset_yolo")

print("Modelo:", modelo_path.resolve())
print("Dataset:", dataset_dir.resolve())

# cargar el mejor modelo
model = YOLO(str(modelo_path))

# evaluar con las imagenes de test
resultados_test = model.val(
    data=str(dataset_dir / "data.yaml"),
    split="test",
    imgsz=640,
    batch=8,
    device=0,
    plots=True,
    project=str(dataset_dir.parent / "runs"),
    name="test_matriculas"
)


Modelo: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt
Dataset: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo
Ultralytics 8.4.191  Python-3.9.21 torch-2.4.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2060, 6144MiB)
YOLO11n summary (fused): 101 layers, 2,597,051 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.1±0.0 ms, read: 952.5±468.9 MB/s, size: 2168.7 KB)
val: Scanning F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\labels\test... 146 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 146/146 90.5it/s 1.6s0.4s
val: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\images\test\20261031_191028.jpg: corrupt JPEG restored and saved
val: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\matriculas\dataset_yolo\images\test\PXL_20261101_1420107

In [25]:

import cv2
from pathlib import Path
from ultralytics import YOLO

# buscar el mejor modelo entrenado
modelos = list(Path(".").rglob("train_matriculas/weights/best.pt"))

if not modelos:
    modelos = list(Path("..").rglob("train_matriculas/weights/best.pt"))

if not modelos:
    raise FileNotFoundError("No se encuentra best.pt")

modelo_path = modelos[0]

# video de entrada
video_path = Path("C0142.MP4")

if not video_path.exists():
    video_path = Path("P4/C0142.MP4")

if not video_path.exists():
    raise FileNotFoundError("No se encuentra C0142.MP4")

# cargar modelo
model = YOLO(str(modelo_path))

# abrir video
cap = cv2.VideoCapture(str(video_path))

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir el video")

fps = cap.get(cv2.CAP_PROP_FPS)
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# crear video de salida
salida_path = video_path.parent / (video_path.stem + "_matriculas.mp4")

writer = cv2.VideoWriter(
    str(salida_path),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (ancho, alto)
)

# procesar video
total_detecciones = 0
fotograma = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    # detectar matriculas
    resultados = model.predict(
        frame,
        imgsz=640,
        conf=0.25,
        verbose=False
    )

    # dibujar detecciones
    for box in resultados[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        confianza = float(box.conf[0])

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        cv2.putText(
            frame,
            f"Matricula {confianza:.2f}",
            (x1, max(y1 - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2
        )

        total_detecciones += 1

    writer.write(frame)
    fotograma += 1

    if fotograma % 100 == 0:
        print(f"Procesados: {fotograma}/{total_frames}")

# liberar recursos
cap.release()
writer.release()

print("\nProcesamiento terminado")
print("Fotogramas:", fotograma)
print("Detecciones totales:", total_detecciones)
print("Video generado:", salida_path.resolve())


Procesados: 100/2832
Procesados: 200/2832
Procesados: 300/2832
Procesados: 400/2832
Procesados: 500/2832
Procesados: 600/2832
Procesados: 700/2832
Procesados: 800/2832
Procesados: 900/2832
Procesados: 1000/2832
Procesados: 1100/2832
Procesados: 1200/2832
Procesados: 1300/2832
Procesados: 1400/2832
Procesados: 1500/2832
Procesados: 1600/2832
Procesados: 1700/2832
Procesados: 1800/2832
Procesados: 1900/2832
Procesados: 2000/2832
Procesados: 2100/2832
Procesados: 2200/2832
Procesados: 2300/2832
Procesados: 2400/2832
Procesados: 2500/2832
Procesados: 2600/2832
Procesados: 2700/2832
Procesados: 2800/2832

Procesamiento terminado
Fotogramas: 2832
Detecciones totales: 2234
Video generado: F:\Ingenieria Informática - ULPGC\QUINTO - ASIGNATURAS\VC\VC_P4\VC_P4\P4\C0142_matriculas.mp4


In [ ]:

import cv2
import csv
import time
from ultralytics import YOLO
import easyocr
from pathlib import Path

# 1. Localización robusta del modelo 'best.pt'
posibles_rutas = [
    Path("runs/matriculas/runs/train_matriculas/weights/best.pt"),
    Path("../train_matriculas/weights/best.pt"),
    Path("best.pt")
]

modelo_path = None
for p in posibles_rutas:
    if p.exists():
        modelo_path = p.absolute()
        break

if modelo_path is None:
    raise FileNotFoundError("No se encuentra el archivo 'best.pt'. Comprueba la ruta de tu modelo.")

print(f">> Cargando modelo desde: {modelo_path}")
model_yolo = YOLO(str(modelo_path))

# Inicializar EasyOCR en modo bruto (sin preprocesamientos)
print(">> Inicializando EasyOCR...")
reader_easyocr = easyocr.Reader(['es', 'en'], gpu=False)

# Diccionario para almacenar el mejor resultado por cada ID de seguimiento
resultados_por_id = {}

# 2. Configurar vídeo
video_path = "C0142.mp4"
cap = cv2.VideoCapture(str(video_path))
# empezar en segundo 50
cap.set(cv2.CAP_PROP_POS_MSEC, 50 * 1000)  # 50 segundos en milisegundos

if not cap.isOpened():
    print(f">> Error: No se pudo abrir el vídeo '{video_path}'.")
else:
    print(f">> Procesando vídeo. Pulsa 'q' o 'ESC' para terminar anticipadamente.")

frame_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1
    
    # Usamos tracking con BoT-SORT para mayor persistencia y evitar saltos extraños de ID
    results = model_yolo.track(frame, persist=True, tracker="botsort.yaml", verbose=False)

    for r in results:
        boxes = r.boxes
        if len(boxes) > 0:
            track_ids = boxes.id.int().cpu().tolist() if boxes.id is not None else list(range(len(boxes)))
            
            for box, track_id in zip(boxes, track_ids):
                cls_id = int(box.cls[0])
                cls_name = model_yolo.names[cls_id]
                conf = float(box.conf[0])
                
                # Coordenadas del bounding box
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # Recorte ROI DIRECTAMENTE EN BRUTO (sin filtros ni reescalados)
                roi = frame[y1:y2, x1:x2]
                # añadir algun filtro "HACER" se pueden filtros que pueden ya estar hechos

                clean_text = ""
                ocr_confidence = 0.0
                ocr_time_ms = 0.0
                
                if roi.size > 0:
                    # mejorar detalles de la imagen
                    roi = cv2.detailEnhance(roi, sigma_s=10, sigma_r=0.15)

                    # ampliar la imagen
                    roi = cv2.resize(
                        roi, None,
                        fx=5, fy=5,
                        interpolation=cv2.INTER_CUBIC
                    )

                    cv2.imshow("ROI", roi)

                    start_time = time.time()
                    # pasar toda la ROI como una unica region de texto para que EasyOCR 
                    # se centre directamente en reconocer los caracteres, sin tener que localizarlos primero
                    h, w = roi.shape[:2]

                    ocr_results = reader_easyocr.recognize(
                        roi,
                        horizontal_list=[[0, w, 0, h]],
                        free_list=[],
                        detail=1,
                        paragraph=False
                    )
                    ocr_time_ms = (time.time() - start_time) * 1000
                    
                    if ocr_results:
                        # ordenar los resultados de izquierda a derecha
                        ocr_results.sort(key=lambda res: min(point[0] for point in res[0]))

                        text_detected = "".join([res[1] for res in ocr_results])
                        ocr_confidence = sum([res[2] for res in ocr_results]) / len(ocr_results)

                        # Limpieza básica alfanumérica
                        clean_text = "".join(e for e in text_detected if e.isalnum()).upper()

                # Si no detectó nada, guardamos una etiqueta indicándolo para la comparativa
                texto_a_guardar = clean_text if clean_text else "SIN_TEXTO"

                # Consolidar por ID: nos quedamos con el registro que tenga texto real o mayor confianza
                if track_id not in resultados_por_id:
                    resultados_por_id[track_id] = {
                        'clase': cls_name,
                        'confianza_yolo': conf,
                        'confianza_ocr': ocr_confidence,
                        'texto': texto_a_guardar,
                        'frame': frame_count,
                        'tiempo_ocr': ocr_time_ms
                    }
                else:
                    current_best = resultados_por_id[track_id]

                    # guardar únicamente lecturas reales con mayor confianza OCR
                    if texto_a_guardar != "SIN_TEXTO" and ocr_confidence > current_best['confianza_ocr']:
                        resultados_por_id[track_id] = {
                            'clase': cls_name,
                            'confianza_yolo': conf,
                            'confianza_ocr': ocr_confidence,
                            'texto': texto_a_guardar,
                            'frame': frame_count,
                            'tiempo_ocr': ocr_time_ms
                        }

                # Visualización en pantalla
                display_text = resultados_por_id[track_id]['texto']
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(frame, f"ID:{track_id} | {display_text}", (x1, max(y1 - 10, 20)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Mostrar vídeo
    cv2.imshow("P4b - Tracking y OCR en Bruto", frame)
    
    # Salida inmediata con 'q' o 'ESC'
    key = cv2.waitKey(1) & 0xFF
    if key == ord('q') or key == 27:
        print(">> Interrumpido por el usuario. Guardando datos hasta el momento...")
        break

cap.release()
cv2.destroyAllWindows()

# 3. Guardado en CSV consolidado
csv_filename = 'resultados_comparativa_easyocr.csv'
print(f"\n>> Guardando {len(resultados_por_id)} registros únicos en '{csv_filename}'...")

with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Track_ID', 'Clase', 'Mejor_Frame', 'Confianza_YOLO', 'Confianza_OCR', 'Matricula_Texto', 'Tiempo_OCR_ms'])
    
    for track_id, data in resultados_por_id.items():
        csv_writer.writerow([
            track_id,
            data['clase'],
            data['frame'],
            round(data['confianza_yolo'], 2),
            round(data['confianza_ocr'], 2),
            data['texto'],
            round(data['tiempo_ocr'], 2)
        ])

print(">> ¡Proceso completado con éxito!")


>> Cargando modelo desde: c:\Users\paull\Documents\ULPGC\VC\P4\VC_P4\P4\runs\matriculas\runs\train_matriculas\weights\best.pt


Using CPU. Note: This module is much faster with a GPU.


>> Inicializando EasyOCR...
>> Procesando vídeo. Pulsa 'q' o 'ESC' para terminar anticipadamente.
>> Interrumpido por el usuario. Guardando datos hasta el momento...

>> Guardando 13 registros únicos en 'resultados_comparativa_easyocr.csv'...
>> ¡Proceso completado con éxito!
